# Darukaa Adaptive Biodiversity Assessment — Nandoshi Lake

**Purpose:** complete Colab workflow for the aquatic/lake profile, from repository installation and Google Earth Engine authentication through geometry QA, automatically derived water domains, aquatic metrics, readiness checks, provenance, and baseline/monitoring comparison.

### Critical spatial rule
The uploaded KML is the **fixed master assessment boundary**. The pipeline does **not** require a hand-drawn water mask. Water surface is derived dynamically from Earth observation for each analysis period. The same KML is not treated as the ecological mask for every metric.

### Output philosophy
Raw measurements, temporal windows, spatial domains, dataset names, observation counts and QA flags are retained. Aquatic proxies without an approved ecological threshold are **not silently converted into a composite concern score**. Composite aquatic State-of-Nature scoring is disabled by default.

## 0. Reproducibility checklist

For a client-grade run, pin the repository to an exact Git commit, retain the uploaded KML, record the GEE project, keep the generated `assessment_manifest.json`, and use the same profile/configuration for baseline and monitoring runs. The old reference package is preserved under `legacy/` and is not overwritten by this lake profile.

In [ ]:
# Configuration for the Colab run
REPO_URL = "https://github.com/G-auravSingh/reference-benchmarking.git"
GIT_REF = "main"  # For final/client runs, replace with an exact commit SHA.
REPO_DIR = "/content/reference-benchmarking"
PROFILE_PATH = f"{REPO_DIR}/profiles/aquatic_lake.yaml"
GEE_PROJECT = "darukaa-earth-product"  # Change only if your EE project differs.


In [ ]:
# Clone/update the repository and install dependencies.
# This cell is intentionally explicit so the same notebook can be reproduced on a clean Colab runtime.
import os, subprocess, sys, pathlib

def run_cmd(cmd, cwd=None):
    print("$", " ".join(cmd))
    r=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(r.stdout)
    r.check_returncode()

if not os.path.exists(REPO_DIR):
    run_cmd(["git","clone","--depth","1","--branch",GIT_REF,REPO_URL,REPO_DIR])
else:
    print("Repository directory already exists:", REPO_DIR)
    run_cmd(["git","status","--short"], cwd=REPO_DIR)

run_cmd([sys.executable,"-m","pip","install","-q","-r",f"{REPO_DIR}/requirements.txt"])
run_cmd([sys.executable,"-m","pip","install","-q","-e",REPO_DIR])


In [ ]:
# Record the exact Git commit visible in the runtime.
import subprocess
try:
    git_sha=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
except Exception:
    git_sha="unknown"
print("Repository commit:", git_sha)
print("NOTE: Replace GIT_REF with this SHA for a fixed reproducible client run.")


## 1. Google Earth Engine authentication

In [ ]:
import ee

try:
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine is already initialized:", GEE_PROJECT)
except Exception as e:
    print("Authentication/initialization required:", e)
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine initialized:", GEE_PROJECT)


In [ ]:
# Confirm that the current public datasets used by the lake profile can be opened.
checks = {
    "Dynamic World": "GOOGLE/DYNAMICWORLD/V1",
    "Sentinel-2 SR Harmonized": "COPERNICUS/S2_SR_HARMONIZED",
    "Sentinel-1 GRD": "COPERNICUS/S1_GRD",
    "JRC Global Surface Water (historical context)": "JRC/GSW1_4/GlobalSurfaceWater",
}
for label, asset in checks.items():
    try:
        obj = ee.ImageCollection(asset) if asset.startswith(("GOOGLE/", "COPERNICUS/")) and asset != "JRC/GSW1_4/GlobalSurfaceWater" else ee.Image(asset)
        print(f"✓ {label}: {asset}")
    except Exception as e:
        print(f"✗ {label}: {asset} -> {e}")


**Dataset note:** Dynamic World is a 10 m near-real-time land-cover product; Sentinel-2 SR Harmonized is available from 2017 onward; Sentinel-1 GRD provides radar observations; JRC Global Surface Water v1.4 contains mapped water history through 2021 and is therefore treated only as historical context in this release.

## 2. Upload the master project KML

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Upload the project KML/KMZ to continue.")

allowed={".kml",".kmz"}
site_candidates=[Path(name) for name in uploaded if Path(name).suffix.lower() in allowed]
if not site_candidates:
    raise ValueError("No .kml or .kmz file was uploaded.")
SITE_FILE=str(site_candidates[0])
print("Using site boundary:", SITE_FILE)


## 3. Load geometry and run spatial QA

In [ ]:
from darukaa_adaptive.site import read_kml, area_ha, make_domains, geometry_hash

geom, named_parts = read_kml(SITE_FILE)
print("Named geometry parts:", len(named_parts))
print("Part names:", list(named_parts)[:20])
print(f"Master boundary area: {area_ha(geom):.4f} ha")
print("KML SHA-256:", geometry_hash(SITE_FILE))
print("Bounds:", tuple(round(x,6) for x in geom.bounds))


In [ ]:
# Load the profile configuration.
from darukaa_adaptive.config import AssessmentConfig

cfg=AssessmentConfig.from_yaml(PROFILE_PATH)
cfg.gee.project_id=GEE_PROJECT
print(cfg.to_dict())
errors=cfg.validate()
if errors:
    raise ValueError(errors)
print("✓ configuration validated")


In [ ]:
# Create the fixed spatial domains.
from darukaa_adaptive.site import ee_geometry

domains=make_domains(geom,cfg.spatial.riparian_buffer_m,cfg.spatial.context_buffer_km)
print("Fixed domains created:")
print("  master boundary")
print(f"  riparian buffer: {cfg.spatial.riparian_buffer_m:.0f} m outside master boundary")
print(f"  context: {cfg.spatial.context_buffer_km:.1f} km outside master boundary")
print("Dynamic water domain is generated later from EO observations.")


### Spatial interpretation
The pipeline cannot infer whether the KML was originally intended to represent a lake outline, project boundary, intervention area, or another feature. In the absence of metadata, this release treats it as the **master assessment boundary** and records that assumption in the manifest.

A dynamic water mask is therefore derived *inside that master boundary* and can vary by date/period. Exposed pixels are not automatically labelled as terrestrial habitat.

## 4. Interactive view of the fixed domains

In [ ]:
import geemap

m=geemap.Map()
m.centerObject(domains["boundary"], 14)
m.addLayer(domains["boundary"], {"color":"red"}, "Master boundary")
m.addLayer(domains["riparian_fixed"], {"color":"yellow"}, "Fixed 100 m riparian ring")
m.addLayer(domains["context"], {"color":"cyan"}, "5 km context")
m


## 5. Dynamic water detector

In [ ]:
from darukaa_adaptive.water import WaterDetector
water=WaterDetector(cfg)

# A full-year period avoids using a partial current year in a Year-0-style baseline.
RUN_START=f"{cfg.temporal.start_year}-01-01"
RUN_END=f"{cfg.temporal.end_year+1}-01-01"
print("Analysis window:", RUN_START, "to", RUN_END)
print("Primary method:", cfg.water.primary_dataset)
print("Primary water probability threshold:", cfg.water.primary_probability_threshold)


In [ ]:
# Dynamic water extent for the configured analysis window.
water_summary=water.area_summary(domains["boundary"],RUN_START,RUN_END)
water_summary


### The important quantity
`water_fraction_pct` is the EO-derived water area as a percentage of the fixed master boundary for the requested period. It is a **monitoring quantity**, not a universal “higher is better” condition score because lake hydroperiod is system-specific.

## 6. Monthly dynamic-water series for a full calendar year

In [ ]:
# Monthly water extent is derived automatically; no manual water polygons are used.
from calendar import monthrange

def month_periods(year):
    out=[]
    for month in range(1,13):
        start=f"{year}-{month:02d}-01"
        end_year=year if month<12 else year+1
        end_month=month+1 if month<12 else 1
        end=f"{end_year}-{end_month:02d}-01"
        out.append({"label":f"{year}-{month:02d}","start":start,"end":end})
    return out

year_for_series=cfg.temporal.end_year
monthly=[]
for p in month_periods(year_for_series):
    r=water.area_summary(domains["boundary"],p["start"],p["end"])
    r["period"]=p["label"]
    monthly.append(r)

import pandas as pd
monthly_df=pd.DataFrame(monthly)
monthly_df[["period","water_area_ha","water_fraction_pct","primary_images","method"]]


In [ ]:
import matplotlib.pyplot as plt

ax=monthly_df.plot(x="period",y="water_fraction_pct",kind="line",marker="o",figsize=(12,4))
ax.set_ylabel("Mean EO water area (% of master boundary)")
ax.set_xlabel("Month")
ax.set_title(f"Dynamic water extent — {year_for_series}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Visualize one automatically derived water mask

In [ ]:
selected_start=f"{year_for_series}-09-01"
selected_end=f"{year_for_series}-10-01"
mask,method,n_images=water.water_mask_for_period(domains["boundary"],selected_start,selected_end)
print("Period:",selected_start,selected_end,"| method:",method,"| observations:",n_images)

mw=geemap.Map()
mw.centerObject(domains["boundary"],14)
mw.addLayer(domains["boundary"],{"color":"red"},"Master boundary")
mw.addLayer(mask.selfMask(),{"palette":["0000FF"]},"Dynamic water mask")
mw


The blue layer in the map is produced by the detector from Earth Engine imagery for the selected period. It is not a manually supplied secondary geometry.

## 8. Long-window water persistence

In [ ]:
persistence=water.persistence(domains["boundary"],RUN_START,RUN_END)
persistence


Water persistence is the spatial mean of the fraction of valid Dynamic World observations classified as water. It should be interpreted together with the observation count and the analysis window.

## 9. Run the lake metric suite

In [ ]:
from darukaa_adaptive.metrics import LakeMetrics
metrics_engine=LakeMetrics(cfg,water)

metric_results=metrics_engine.run(domains["boundary"],domains["riparian_fixed"])
metric_df=pd.DataFrame([m.to_dict() for m in metric_results])
metric_df[["metric","pillar","domain","value","units","status","temporal_window","dataset","scale_m","direction","score_eligible"]]


### Interpretation guardrail
The output distinguishes `value` from `score_eligible`. A metric can be ecologically useful and still be intentionally left out of a composite score when no validated general threshold is configured.

## 10. Water-quality proxy inspection

In [ ]:
display_cols=["metric","value","units","domain","status","valid_observations","notes"]
metric_df[metric_df["metric"].isin([
    "ndci_proxy","red_reflectance_turbidity_proxy","surface_algal_bloom_frequency"
])][display_cols]


NDCI and FAI-derived values are optical proxies. Their raw values can be used for repeat monitoring with identical processing, but a project should not turn them into a universal “poor/good” score without calibration to local water conditions or an externally approved threshold protocol.

## 11. Standardized riparian disturbance

In [ ]:
metric_df[metric_df["metric"]=="shoreline_disturbance_fraction"][display_cols]


This indicator uses the automatically generated fixed 100 m riparian ring outside the master boundary. It reports the fraction mapped as crops, built area, or bare ground. It is a contextual pressure metric, not a direct measurement of biodiversity loss.

## 12. Multi-year riparian NDVI trend

In [ ]:
metric_df[metric_df["metric"]=="riparian_ndvi_sen_slope"][display_cols]


The trend uses annual seasonal composites and a Theil–Sen slope. Kendall tau significance is retained in the metric notes. A minimum temporal-depth gate prevents a short two-year series from being described as a long-term trend.

## 13. Readiness checks

In [ ]:
from darukaa_adaptive.readiness import assess_readiness
readiness=assess_readiness(cfg,area_ha(geom),metric_results)
readiness


The readiness block intentionally separates measurement readiness from ecological validation. Remote sensing can produce repeatable proxies without proving that those proxies are calibrated measures of the biological process of interest.

## 14. Produce auditable output files

In [ ]:
from darukaa_adaptive.report import write_assessment

output_dir=cfg.output_dir
paths=write_assessment(
    output_dir=output_dir,
    config=cfg,
    site_path=SITE_FILE,
    boundary_area_ha=area_ha(geom),
    domains=domains,
    metrics=metric_results,
    water_periods=[water_summary],
    readiness=readiness,
)
paths


In [ ]:
# Save the monthly water series alongside the standard outputs.
from pathlib import Path
Path(output_dir).mkdir(exist_ok=True,parents=True)
monthly_df.to_csv(Path(output_dir)/"water_monthly.csv",index=False)
print("Output directory contents:")
for p in sorted(Path(output_dir).iterdir()):
    print(" -",p)


## 15. Baseline storage for future monitoring

In [ ]:
# Use this cell for the Year-0 baseline run. It makes a clearly named copy for future comparisons.
from shutil import copyfile
BASELINE_COPY=str(Path(output_dir)/"baseline_metric_scorecard.csv")
copyfile(paths["metric_scorecard"],BASELINE_COPY)
print("Baseline scorecard saved:",BASELINE_COPY)


## 16. Future monitoring: compare with the same metric definitions

In [ ]:
# For a Year-1+ run, upload the baseline_metric_scorecard.csv from the Year-0 run.
from google.colab import files

print("Upload the baseline_metric_scorecard.csv created by this same adaptive pipeline.")
base_upload=files.upload()
if base_upload:
    BASELINE_FILE=next(name for name in base_upload if name.endswith(".csv"))
    print("Using:",BASELINE_FILE)
else:
    BASELINE_FILE=None


In [ ]:
if BASELINE_FILE:
    from darukaa_adaptive.trajectory import compare
    current_csv=paths["metric_scorecard"]
    traj=compare(current_csv,BASELINE_FILE)
    traj


The comparison is intentionally descriptive: it reports current value, baseline value, absolute change and percent change. A raw change is not automatically labelled as ecological recovery because directionality may be context-dependent (for example, water extent).

## 17. Composite State-of-Nature scoring — deliberately gated

In [ ]:
from darukaa_adaptive.scoring import summarize_composite

composite=summarize_composite(metric_results, thresholds_by_metric={})
print(composite)


### Why the composite is disabled

The previous workflow could average populated five-class concern values even when some aquatic indicators were proxies, some were not applicable, or the time/data requirements were not met. This release does not invent generic aquatic thresholds.

An approved threshold set can be supplied later using the exact `t1 < t2 < t3 < t4` interface in `darukaa_adaptive.scoring.score_with_thresholds()`, but only after those thresholds are scientifically reviewed for the intended metric, ecosystem, sensor, and temporal window.

## 18. Legacy-vs-adaptive audit note

The adaptive lake profile does **not** replace `darukaa_reference_v0.1.0`. The source-level audit is documented in `docs/REFERENCE_PIPELINE_AUDIT.md`, and the exact legacy package is retained under `legacy/`. This separation is required so that previous terrestrial reports remain reproducible while the aquatic workflow can use explicit dynamic spatial domains and aquatic-specific logic.

## 19. Download the results

In [ ]:
# Package the generated outputs for local download.
import shutil, os
zip_base="darukaa_nandoshi_aquatic_outputs"
archive=shutil.make_archive(zip_base,"zip",output_dir)
print("Created:",archive)

from google.colab import files
files.download(archive)


## Troubleshooting

**Earth Engine authentication fails:** confirm that the Google account has Earth Engine access and that `GEE_PROJECT` is a valid project permitted to use EE.

**Dynamic World returns too few images:** the code automatically attempts Sentinel-1 VV fallback when enabled. The result records the method used.

**Riparian trend says insufficient temporal depth:** extend `temporal.start_year` backward or use a project-approved monitoring window; do not override the minimum simply to force a trend.

**Metric value is `None`:** inspect the `status`, observation count, and domain. A missing value is preferable to silently substituting an unrelated mask or threshold.

**Do not compare the old report's values directly to new values as a time series** unless the metric definition, spatial domain, dataset, temporal window and aggregation are identical.